In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import find_peaks

# ============================================================
# Global Spillover Spike Detection from GFEVD
# ============================================================

BASE_DIR = Path("./")
RESULT_DIR = Path("./results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CSV = BASE_DIR / "tvpvar_input_scaled.csv"
INPUT_GFEVD = BASE_DIR / "gfevd_all.npy"

DATE_COL = "Date"
TARGET_VAR = "dlog_SOLVPN"

Z_K = 2.326
MIN_DISTANCE = 5
LOCAL_PEAK = True

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(INPUT_CSV)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

features = [c for c in df.columns if c != DATE_COL]

gfevd = np.load(INPUT_GFEVD)

print("Input rows:", len(df))
print("GFEVD shape:", gfevd.shape)
print("Features:", features)

# gfevd shape = (T, N, N)
T, N, _ = gfevd.shape

if N != len(features):
    raise ValueError("Feature count and GFEVD dimension do not match.")

# lag 차이 보정
dates = df[DATE_COL].iloc[-T:].reset_index(drop=True)

# -----------------------------
# Build pairwise spillover
# convention:
# gfevd[t, i, j] = contribution from shock j to variable i
# j -> i
# -----------------------------
records = []

target_idx = features.index(TARGET_VAR)

for other in features:
    if other == TARGET_VAR:
        continue

    other_idx = features.index(other)

    solvpn_to_other = gfevd[:, other_idx, target_idx]
    other_to_solvpn = gfevd[:, target_idx, other_idx]
    net_solvpn_to_other = solvpn_to_other - other_to_solvpn

    temp = pd.DataFrame({
        "Date": dates,
        "Pair": f"{TARGET_VAR}__{other}",
        "Source": TARGET_VAR,
        "Target": other,
        "SOLVPN_to_X": solvpn_to_other,
        "X_to_SOLVPN": other_to_solvpn,
        "PairwiseNet_SOLVPN_to_X": net_solvpn_to_other
    })

    records.append(temp)

pairwise_df = pd.concat(records, ignore_index=True)

pairwise_out = RESULT_DIR / "solvpn_pairwise_spillover_from_gfevd.csv"
pairwise_df.to_csv(pairwise_out, index=False, encoding="utf-8-sig")

print(f"Saved pairwise spillover: {pairwise_out}")

# -----------------------------
# Global spike detection
# -----------------------------
spike_records = []

for pair, sub in pairwise_df.groupby("Pair"):
    for col in ["SOLVPN_to_X", "X_to_SOLVPN", "PairwiseNet_SOLVPN_to_X"]:
        s = sub[col].astype(float).reset_index(drop=True)

        mean = s.mean()
        std = s.std()
        threshold = mean + Z_K * std

        if std == 0 or np.isnan(std):
            continue

        if LOCAL_PEAK:
            peak_idx, _ = find_peaks(
                s.values,
                height=threshold,
                distance=MIN_DISTANCE
            )
            spike_idx = peak_idx
        else:
            spike_idx = np.where(s.values > threshold)[0]

        for idx in spike_idx:
            value = s.iloc[idx]
            z_value = (value - mean) / std

            spike_records.append({
                "Date": sub["Date"].iloc[idx],
                "Pair": pair,
                "DirectionMetric": col,
                "Spillover": value,
                "GlobalMean": mean,
                "GlobalStd": std,
                "GlobalThreshold": threshold,
                "GlobalZ": z_value,
                "SpikeType": "global"
            })

global_spikes = pd.DataFrame(spike_records)

if len(global_spikes) > 0:
    global_spikes = global_spikes.sort_values(
        ["Date", "Pair", "DirectionMetric"]
    ).reset_index(drop=True)

spike_out = RESULT_DIR / "global_spillover_spikes_SOLVPN.csv"
global_spikes.to_csv(spike_out, index=False, encoding="utf-8-sig")

print(f"Saved global spikes: {spike_out}")
print("Number of global spikes:", len(global_spikes))
print(global_spikes.head(30))

Input rows: 606
GFEVD shape: (605, 10, 10)
Features: ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_GOLD', 'dlog_SILVER', 'dlog_DXY', 'dlog_OIL', 'dlog_GAS', 'd_UST10Y', 'd_VIX', 'd_PJM_LMP']
Saved pairwise spillover: results\solvpn_pairwise_spillover_from_gfevd.csv
Saved global spikes: results\global_spillover_spikes_SOLVPN.csv
Number of global spikes: 131
         Date                      Pair          DirectionMetric  Spillover  \
0  2022-12-07     dlog_SOLVPN__dlog_OIL              X_to_SOLVPN   0.317733   
1  2022-12-13    dlog_SOLVPN__d_PJM_LMP              SOLVPN_to_X   0.176942   
2  2022-12-13     dlog_SOLVPN__dlog_GAS              SOLVPN_to_X   0.160379   
3  2022-12-13     dlog_SOLVPN__dlog_OIL              SOLVPN_to_X   0.201557   
4  2022-12-15    dlog_SOLVPN__d_PJM_LMP              X_to_SOLVPN   0.240983   
5  2022-12-19    dlog_SOLVPN__dlog_GOLD              X_to_SOLVPN   0.204482   
6  2022-12-21     dlog_SOLVPN__dlog_OIL              X_to_SOLVPN   0.147677   
7  2022-12-27    d